In [30]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

df_orders= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_orders_dataset.csv')
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [31]:
df_customers= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_customers_dataset.csv')
df_order_items= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_order_items_dataset.csv')
df_order_payments= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_order_payments_dataset.csv')  
df_products= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_products_dataset.csv')
df_sellers= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_sellers_dataset.csv')
df_geolocation= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_geolocation_dataset.csv')
df_reviews= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/olist_order_reviews_dataset.csv')  
df_product_category_name= pd.read_csv('C:/Users/HP/Desktop/E-commerce/Data/product_category_name_translation.csv')

In [32]:
# 1A. Deduplicate Geolocation to 1 row per zip code prefix
geo_clean = (
    df_geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        lat=('geolocation_lat', 'mean'),
        lng=('geolocation_lng', 'mean')
    )
)

In [33]:
# 1B. Aggregate Payments to 1 row per order_id
payments_agg = (
    df_order_payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value=('payment_value', 'sum'),
        payment_installments=('payment_installments', 'max'),
        primary_payment_type=('payment_type', 'first'),
        payment_count=('payment_sequential', 'count')
    )
)

In [34]:
# 1C. Aggregate Order Items to 1 row per order_id (for Order-Level Master)
items_agg = (
    df_order_items
    .groupby('order_id', as_index=False)
    .agg(
        total_items_price=('price', 'sum'),
        total_freight_value=('freight_value', 'sum'),
        total_items_count=('order_item_id', 'count'),
        unique_sellers_count=('seller_id', 'nunique')
    )
)

In [35]:
# 1D. Deduplicate Reviews to 1 row per order_id (keep latest review)
reviews_clean = (
    df_reviews
    .sort_values('review_creation_date', ascending=False)
    .drop_duplicates(subset=['order_id'], keep='first')
    [['order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date']]
)

In [36]:
# 1E. Enrich Products with English Category Names
products_clean = df_products.merge(
    df_product_category_name,
    on='product_category_name',
    how='left'
)
products_clean['product_category_name_english'] = products_clean['product_category_name_english'].fillna('uncategorized')


In [ ]:
# STEP 2: Build Master Table 1 — ORDER-LEVEL (1 row = 1 Order)
# Best for: Revenue Reconciliation, Delivery SLAs, Cohort Retention
df_order_master = (
    df_orders
    .merge(df_customers, on='customer_id', how='left', validate='1:1')
    .merge(geo_clean, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left', validate='m:1')
    .merge(items_agg, on='order_id', how='left', validate='1:1')
    .merge(payments_agg, on='order_id', how='left', validate='1:1')
    .merge(reviews_clean, on='order_id', how='left', validate='1:1')
)

In [38]:
# Verify zero row inflation (must match len(df_orders) = 99,441)
assert len(df_order_master) == len(df_orders), "Row count mismatch in Order Master!"
print(f"Order Master Shape: {df_order_master.shape}")

Order Master Shape: (99441, 27)


In [39]:
# STEP 3: Build Master Table 2 — ITEM-LEVEL (1 row = 1 Line Item)
# Best for: Product Category Analysis, Seller Scorecards, Freight Ratios
# ---------------------------------------------------------
df_item_master = (
    df_order_items
    .merge(df_orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date']], on='order_id', how='left', validate='m:1')
    .merge(products_clean[['product_id', 'product_category_name_english', 'product_weight_g']], on='product_id', how='left', validate='m:1')
    .merge(df_sellers, on='seller_id', how='left', validate='m:1')
)

# Verify zero row inflation (must match len(df_order_items) = 112,650)
assert len(df_item_master) == len(df_order_items), "Row count mismatch in Item Master!"
print(f"Item Master Shape: {df_item_master.shape}")

Item Master Shape: (112650, 16)
